# Семинар 4. Граф, ветвления и параллельность

**Python 3.10+ · только стандартная библиотека**

Это автономный учебный ноутбук: загрузите файл в Colab либо Jupyter и выполняйте ячейки сверху вниз.
Установка пакетов, ключи, GPU и сетевой доступ не требуются. Здесь используется **офлайн-симуляция**, а не настоящая LLM.

Предпосылки: функции, словари, исключения и базовые тесты `assert`; материал предыдущих занятий.
Четыре ячейки с тегом `exercise` содержат задания. После заполнения каждого задания запускайте его открытые проверки.
Пока функция не реализована, её вызов завершится `NotImplementedError`.

Баллы: 2 за каждое задание, 2 за корректный эксперимент и разбор ограничений.

Критерий сдачи: четыре функции, пройденные проверки, один дополнительный отрицательный случай и ответы в итоговом отчёте.
Результаты симуляции нельзя подписывать как качество или скорость языковой модели.


Построим DAG и проверим три разные вещи: корректность зависимостей, согласование свидетельств и расчёт расписания.
    Маршрутизатор задан правилами; LLM отсутствует. Потоки выполняют обычные учебные функции.
    Длительности операций в финальном эксперименте **синтетические**, это не измерение скорости API.

    Чтение/сравнение: [Graph API LangGraph](https://docs.langchain.com/oss/python/langgraph/graph-api),
    [LLMCompiler](https://arxiv.org/abs/2312.04511), выдержка CARL приложена к лекции.
    Ни CARL, ни LangGraph не импортируются: это самостоятельный учебный исполнитель.


In [1]:
import copy
import json
from concurrent.futures import ThreadPoolExecutor

def expect_error(exception, thunk):
    try:
        thunk()
    except exception:
        return
    raise AssertionError(f"Expected {exception.__name__}")

NODES = [
    {"id": "A", "deps": []}, {"id": "B", "deps": []},
    {"id": "C", "deps": ["A"]}, {"id": "D", "deps": ["B"]},
    {"id": "join", "deps": ["C", "D"]},
]
DURATIONS = {"A": 2, "B": 5, "C": 4, "D": 1, "join": 1}
print("OFFLINE: synthetic DAG and durations")


OFFLINE: synthetic DAG and durations


### Решённый пример: независимые результаты
    Каждый работник возвращает собственный словарь. Объединение восстанавливает порядок по идентификатору.
    Не считайте порядок завершения источником истинности.


In [2]:
def read_fixture(doc_id):
    fixtures = {"doc-a": "314", "doc-b": "314"}
    return {"source": doc_id, "value": fixtures[doc_id], "ok": True}

with ThreadPoolExecutor(max_workers=2) as pool:
    futures = {key: pool.submit(read_fixture, key) for key in ["doc-b", "doc-a"]}
    results = [futures[key].result() for key in sorted(futures)]
assert [r["source"] for r in results] == ["doc-a", "doc-b"]
print(results)


[{'source': 'doc-a', 'value': '314', 'ok': True}, {'source': 'doc-b', 'value': '314', 'ok': True}]


## Задание 1. Проверка плана и слои — 2 балла

`topological_layers(nodes)` возвращает список слоёв, внутри слоя идентификаторы отсортированы.
    nodes — список объектов ровно с id (непустая строка) и deps (список строк).
    Отклоняйте ValueError: повтор id, повтор зависимости, неизвестную зависимость, цикл и неверную форму.
    Пустой граф даёт []. Исходный nodes не меняется. В каждом новом слое только узлы,
    все предшественники которых находятся в **предыдущих** слоях.


In [3]:
def topological_layers(nodes):
    if not nodes:
        return []
    
    deps_map = {}
    id_set = set()
    
    for node in nodes:
        if not isinstance(node, dict) or "id" not in node or "deps" not in node:
            raise ValueError("Неверная форма узла")
        
        n_id = node["id"]
        deps = node["deps"]
        
        if not isinstance(n_id, str) or not n_id:
            raise ValueError("id должен быть непустой строкой")
        if n_id in id_set:
            raise ValueError("Повтор id")
        
        seen_deps = set()
        for d in deps:
            if d in seen_deps:
                raise ValueError("Повтор зависимости")
            seen_deps.add(d)
            
        id_set.add(n_id)
        deps_map[n_id] = deps
        
    for n_id, deps in deps_map.items():
        for d in deps:
            if d not in id_set:
                raise ValueError("Неизвестная зависимость")
                
    layers = []
    placed = set()
    
    while len(placed) < len(deps_map):
        current_layer = []
        for n_id in id_set:
            if n_id not in placed:
                if all(d in placed for d in deps_map[n_id]):
                    current_layer.append(n_id)
        
        if not current_layer:
            raise ValueError("Обнаружен цикл в графе")
            
        current_layer.sort()
        layers.append(current_layer)
        placed.update(current_layer)
        
    return layers

In [4]:
saved = copy.deepcopy(NODES)
assert topological_layers(NODES) == [["A", "B"], ["C", "D"], ["join"]]
assert NODES == saved and topological_layers([]) == []
for bad in ([{"id":"x","deps":["missing"]}], [{"id":"x","deps":["x"]}],
            [{"id":"x","deps":[]},{"id":"x","deps":[]}],
            [{"id":"x","deps":["y"]},{"id":"y","deps":["x"]}]):
    expect_error(ValueError, lambda bad=bad: topological_layers(bad))
print("PASS: topological_layers")


PASS: topological_layers


## Задание 2. Полная маршрутизация — 2 балла

`choose_route(record)` возвращает clarify/direct/search.
    record — словарь. При conflict is True всегда clarify (высший приоритет).
    Иначе direct, если verified is True, answer — непустая строка и source — непустая строка.
    Иначе search, если query — непустая строка. Во всех прочих случаях clarify.
    Строку из пробелов считайте пустой; не принимайте "yes" или число 1 за True.
    Это детерминированные правила, а не модельный классификатор.


In [5]:
def choose_route(record):
    if "conflict" in record and record["conflict"]:
        return "clarify"

    for node in record:
        if isinstance(record[node] , str ) and record[node].count(" ") == len(record[node]):
            return "clarify" 
    if "verified" in record and record["verified"]:
        if "query" in record and len(record["answer"]) != 0 and len(record["source"]) != 0 and len(record["query"]) != 0:
            return "search"
        elif len(record["answer"]) != 0 and len(record["source"]) != 0:
            return "direct" 
        else:
            return "clarify"
    else:
        return "clarify"

    
    raise NotImplementedError("Student exercise")


In [6]:
choose_route({"verified":1,"answer":"314","source":"doc","query":"room"})

'search'

In [7]:
assert choose_route({"conflict":True,"verified":True,"answer":"314","source":"doc"}) == "clarify"
assert choose_route({"verified":True,"answer":"314","source":"doc"}) == "direct"
assert choose_route({"verified":1,"answer":"314","source":"doc","query":"room"}) == "search"
assert choose_route({"query":"   "}) == "clarify"
assert choose_route({"answer":"314"}) == "clarify"
print("PASS: choose_route")


PASS: choose_route


## Задание 3. Объединить, сохранив конфликт — 2 балла

`merge_evidence(items)` использует только элементы с ok is True; остальные пропускает.
    У используемого элемента source и value — непустые строки, иначе ValueError.
    Повторы одинаковой пары источник/значение объединяются; одинаковый источник с разными значениями вызывает ValueError.
    Возвращается {status, value, sources}: при отсутствии данных missing/None/[];
    при одном уникальном значении consistent/значение/отсортированные источники;
    при разных значениях conflict/None/отсортированные источники. Исходные элементы не меняйте.


In [8]:
def merge_evidence(items):
    if items == []:
        return {"status":"missing","value":None,"sources":[]}
    d = []
    d1 = dict()
    for item in items:
        if "ok" in item and item["ok"]:
            if "source" in item and "value" in item and len(item["source"]) != 0 and len(item["value"]) != 0:
                if item["source"] not in d:
                    d.append(item["source"])
                    d1[item["source"]] = item["value"]
                else:
                    if item["value"] != d1[item["source"]]:
                        raise ValueError
                    else:
                        return    {"status":"consistent","value":item["value"],"sources": d} 
                        continue



            else:
                raise ValueError
            
        else:
            return {"status":"missing","value":None,"sources":[]}
            continue 
    if len(d) == 2:
        # print(d1)
        if d1[d[0]] == d1[d[1]]:         
            return {"status":"consistent","value": d1[d[1]],"sources": d}
        else:
           return  {"status":"conflict","value": None,"sources": d}
    else:
        return {"status":"conflict","value": None,"sources": d}


    raise NotImplementedError("Student exercise")


In [9]:
assert merge_evidence([]) == {"status":"missing","value":None,"sources":[]}
one = {"source":"a","value":"314","ok":True}
assert merge_evidence([one, one])["sources"] == ["a"]
assert merge_evidence([one,{"source":"b","value":"314","ok":True}])["status"] == "consistent"
assert merge_evidence([one,{"source":"b","value":"210","ok":True}])["status"] == "conflict"
assert merge_evidence([{"ok":False,"source":"failed","value":"999"}])["status"] == "missing"
expect_error(ValueError, lambda: merge_evidence([one,{"source":"a","value":"210","ok":True}]))
print("PASS: merge_evidence")


PASS: merge_evidence


## Задание 4. Критический путь — 2 балла

`critical_path_time(nodes, durations)` возвращает длину критического пути.
    Используйте проверку графа из задания 1. В durations ровно ключи всех вершин; значения — конечные неотрицательные int/float, не bool.
    Несоответствие вызывает ValueError. Для пустого графа и пустых длительностей результат 0.
    earliest_finish[v] = duration[v] + max(earliest_finish[p] for p in deps[v]), максимум по пустому набору равен 0.
    Это расчёт для неограниченного числа работников без накладных расходов, **не wall-clock benchmark**.


In [10]:
import math

def critical_path_time(nodes, durations):
   
    layers = topological_layers(nodes) 
    
    if not nodes and not durations:
        return 0
        
    node_ids = {n["id"] for n in nodes}
    if node_ids != set(durations.keys()):
        raise ValueError("Несоответствие ключей в графе и durations")
        
    for val in durations.values():
        if isinstance(val, bool):
            raise ValueError("Длительность не может быть bool")
        if not isinstance(val, (int, float)):
            raise ValueError("Длительность должна быть числом")
        if math.isnan(val) or math.isinf(val) or val < 0:
            raise ValueError("Недопустимое значение длительности")
            
    earliest_finish = {}
    deps_map = {n["id"]: n["deps"] for n in nodes}
    
    for layer in layers:
        for n_id in layer:
            deps = deps_map[n_id]
            max_dep_time = max([earliest_finish[d] for d in deps], default=0)
            earliest_finish[n_id] = durations[n_id] + max_dep_time
            
    return max(earliest_finish.values(), default=0)


In [11]:
assert critical_path_time(NODES, DURATIONS) == 7
assert critical_path_time([], {}) == 0
assert critical_path_time([{"id":"a","deps":[]}], {"a":0}) == 0
expect_error(ValueError, lambda: critical_path_time(NODES, {**DURATIONS,"A":True}))
expect_error(ValueError, lambda: critical_path_time(NODES, {**DURATIONS,"A":float("nan")}))
expect_error(ValueError, lambda: critical_path_time(NODES, {"A":1}))
print("PASS: critical_path_time")


PASS: critical_path_time


## Готовый исполнитель по слоям
    Это рабочий stdlib-пример, не задание. Он исполняет каждый слой параллельно, затем ждёт весь слой.
    Работники получают копию результатов своих предшественников. При ошибке работника исключение останавливает запуск;
    уже начатые задачи текущего слоя могут завершиться — отмена внешних эффектов здесь не реализована.


In [12]:
def run_dag(nodes, functions, workers=2):
    if set(functions) != {node["id"] for node in nodes}:
        raise ValueError("function keys differ")
    predecessors = {node["id"]: node["deps"] for node in nodes}
    results = {}
    for layer in topological_layers(nodes):
        with ThreadPoolExecutor(max_workers=workers) as pool:
            futures = {key: pool.submit(functions[key], copy.deepcopy({p: results[p] for p in predecessors[key]})) for key in layer}
            batch = {key: futures[key].result() for key in sorted(layer)}
        results.update(batch)
    return results

functions = {
    "A": lambda inputs: read_fixture("doc-a"),
    "B": lambda inputs: read_fixture("doc-b"),
    "C": lambda inputs: inputs["A"],
    "D": lambda inputs: inputs["B"],
    "join": lambda inputs: merge_evidence([inputs["C"], inputs["D"]]),
}
output = run_dag(NODES, functions)
assert output["join"] == {"status":"consistent","value":"314","sources":["doc-a","doc-b"]}
print(output["join"])


{'status': 'consistent', 'value': '314', 'sources': ['doc-a', 'doc-b']}


## Эксперимент
    Вычислите три величины для одного графа: последовательное время, время слоёв с барьерами, критический путь.
    Сначала нарисуйте расписание на бумаге. В строке эксперимента должны получиться разные значения;
    объясните, почему нельзя подписывать критический путь как замер run_dag.
    Затем поменяйте одну длительность и проверьте прогноз.


In [13]:
serial = sum(DURATIONS.values())
layers = topological_layers(NODES)
barrier = sum(max(DURATIONS[key] for key in layer) for layer in layers)
critical = critical_path_time(NODES, DURATIONS)
print({"mode":"synthetic durations", "serial":serial, "layer_barriers":barrier, "critical_path":critical})
assert (serial, barrier, critical) == (13, 10, 7)
assert critical <= barrier <= serial
changed = {**DURATIONS, "A": 10}
assert critical_path_time(NODES, changed) == 15
print({"changed_A":10, "new_critical_path":critical_path_time(NODES, changed)})


{'mode': 'synthetic durations', 'serial': 13, 'layer_barriers': 10, 'critical_path': 7}
{'changed_A': 10, 'new_critical_path': 15}


## Отчёт
    1. Почему C может начаться раньше завершения B при планировании без общего барьера?
    2. Какие зависимости несут данные, а не просто порядок?
    3. Что произойдёт при конфликте doc-a/doc-b и почему нельзя брать последний ответ?
    4. Какие затраты отсутствуют в синтетических длительностях?
    5. Добавьте отрицательный пример графа либо свидетельств и объясните ожидаемую ошибку.

    Дополнительно: прочитайте приложенную выдержку CARL и документацию LangGraph.
    Сопоставьте id/deps/functions/results с описанием узла, связями, исполнителем и состоянием; укажите, что наш код упрощает.
